# Hospedando agente A2A no Amazon Bedrock AgentCore Runtime - Autenticação de entrada AWS IAM

## Visão geral

Neste tutorial, aprenderemos como hospedar um agente A2A (Agent-to-Agent) no Amazon Bedrock AgentCore Runtime usando AWS IAM para autenticação de entrada.

O [protocolo A2A](https://a2a-protocol.org/dev/specification/) é um padrão aberto projetado para facilitar a comunicação entre sistemas de agentes de IA independentes. Embora o A2A tradicionalmente use tokens OAuth/JWT para autenticação, o AgentCore Runtime permite que você configure credenciais AWS IAM para requisições de entrada, atendendo aos requisitos de segurança empresarial.

### Detalhes do tutorial

| Informação         | Detalhes                                                   |
|:--------------------|:----------------------------------------------------------|
| Tipo de tutorial       | Hospedagem de agente                                             |
| Protocolo            | A2A (Agent-to-Agent)                                      |
| Autenticação      | AWS IAM (SigV4)                                           |
| Componentes do tutorial | Hospedagem de agente A2A no AgentCore Runtime com autenticação IAM     |
| Vertical do tutorial   | Multi-vertical                                            |
| Complexidade do exemplo  | Intermediário                                              |
| SDK usado            | Amazon BedrockAgentCore Python SDK e Strands Agents    |

### Arquitetura do tutorial

Neste tutorial, implantaremos um agente A2A no AgentCore Runtime com autenticação IAM.

Para fins de demonstração, usaremos um agente simples com 2 ferramentas: `greet_user` e `get_agent_info`

### Principais recursos do tutorial

* Criação de agentes A2A com framework Strands
* Teste de agentes A2A localmente
* Hospedagem de agentes A2A no Amazon Bedrock AgentCore Runtime
* Invocação de agentes A2A implantados com autenticação IAM (SigV4)


## Pré-requisitos

Para executar este tutorial você precisará de:
* Python 3.10+
* Credenciais AWS configuradas
* Amazon Bedrock AgentCore SDK
* Framework Strands Agents
* Daemon Docker em execução

In [ ]:
!pip install --force-reinstall -U -r requirements.txt --quiet

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime
from bedrock_agentcore_starter_toolkit.operations.runtime import destroy_bedrock_agentcore
from boto3.session import Session
from pathlib import Path
import os

In [ ]:
boto_session = Session()
region = boto_session.region_name

agentcore_control_client = boto_session.client("bedrock-agentcore-control", region_name=region)

agent_name = "a2a_agent_iam"

## Entendendo A2A (Agent-to-Agent Protocol)

A2A é um protocolo que permite que agentes de IA se comuniquem entre si. Conceitos principais:

* **Agent Card**: Metadados descrevendo as capacidades do agente
* **Troca de mensagens**: Comunicação estruturada entre agentes
* **Ferramentas**: Funções que agentes podem expor para outros agentes
* **Autenticação IAM**: Usando AWS SigV4 para autenticação segura

O AgentCore Runtime espera que agentes A2A sejam hospedados em `0.0.0.0:9000/` como caminho padrão.

### Estrutura do projeto

```
agentcore-a2a-iam-sample/
├── agent.py              # Código principal do agente A2A
├── client.py             # Cliente para teste com autenticação IAM
├── requirements.txt      # Dependências
└── hosting_a2a_iam_auth.ipynb  # Este notebook
```

## Revisar código do agente

Vamos revisar o código do agente que implantaremos:

In [ ]:
!cat agent.py

### O que este código faz

* **Strands Agent**: Cria um agente usando o framework Strands
* **@tool**: Decorador que transforma funções Python em ferramentas de agente
* **A2AServer**: Envolve o agente para suportar o protocolo A2A
* **FastAPI**: Fornece servidor HTTP para o agente
* **Ferramentas**: Duas ferramentas simples demonstrando capacidades do agente

## Opcional: Testando localmente

Antes de implantar no AgentCore Runtime, você pode testar o agente localmente:

1. **Terminal 1**: Inicie o agente
   ```bash
   python agent.py
   ```
   
2. **Terminal 2**: Teste o agent card
   ```bash
   curl http://localhost:9000/.well-known/agent-card.json | jq .
   ```

3. **Terminal 2**: Envie uma mensagem de teste
   ```bash
   curl -X POST http://localhost:9000 \
     -H "Content-Type: application/json" \
     -d '{
       "jsonrpc": "2.0",
       "id": "req-001",
       "method": "message/send",
       "params": {
         "message": {
           "role": "user",
           "parts": [{
             "kind": "text",
             "text": "Olá! O que você pode fazer?"
           }],
           "messageId": "test-001"
         }
       }
     }' | jq .
   ```

## Configurando implantação do AgentCore Runtime

Em seguida, usaremos o starter toolkit para configurar a implantação do AgentCore Runtime. Configuraremos:

* Entrypoint: `agent.py`
* Criar função de execução automaticamente
* Criar repositório ECR automaticamente
* Protocolo: A2A
* Arquivo de requisitos

Durante a etapa de configuração, seu Dockerfile será gerado com base no código da sua aplicação.

In [ ]:
print(f"Using AWS region: {region}")

required_files = ["agent.py", "requirements.txt"]
for file in required_files:
    if not os.path.exists(file):
        raise FileNotFoundError(f"Required file {file} not found")
print("All required files found ✓")

agentcore_runtime = Runtime()

print("Configuring AgentCore Runtime...")
response = agentcore_runtime.configure(
    entrypoint="agent.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    protocol="A2A",
    agent_name=agent_name,
)
print("Configuration completed ✓")

## Iniciando agente A2A no AgentCore Runtime

Agora que temos um Dockerfile, vamos iniciar o agente A2A no AgentCore Runtime. Isso irá:

1. Criar o repositório Amazon ECR
2. Construir e fazer push da imagem Docker
3. Criar o AgentCore Runtime
4. Implantar o agente

Isso pode levar vários minutos...

In [ ]:
print("Launching A2A agent to AgentCore Runtime...")
print("This may take several minutes...")
launch_result = agentcore_runtime.launch()
print("Launch completed ✓")
print(f"Agent ARN: {launch_result.agent_arn}")
print(f"Agent ID: {launch_result.agent_id}")

## Testando seu agente A2A implantado

Agora vamos testar nosso agente A2A implantado usando o cliente com autenticação IAM.

O cliente irá:
1. Usar credenciais AWS para assinar requisições com SigV4
2. Buscar o agent card
3. Enviar mensagens de teste ao agente
4. Exibir respostas

In [ ]:
import os
os.environ['AGENT_ARN'] = launch_result.agent_arn

from client import *

# Test with a custom message
custom_message = "Please greet me. My name is Bob."
await test_agent(launch_result.agent_arn, custom_message)

## Entendendo autenticação IAM

### Como funciona

1. **Lado do cliente**: O cliente usa credenciais AWS para assinar requisições HTTP com SigV4
2. **AgentCore Runtime**: Valida a assinatura usando IAM
3. **Agente**: Recebe requisições autenticadas

### Fluxo de autenticação SigV4

```python
# Cliente cria autenticação SigV4
auth = SigV4HTTPXAuth(credentials, "bedrock-agentcore", region)

# Autenticação assina cada requisição automaticamente
async with httpx.AsyncClient(auth=auth) as client:
    response = await client.post(url, json=data)
```

Para visualizar o código completo do cliente, execute a seguinte célula:

In [ ]:
!cat client.py

## Próximos passos

Agora que você implantou com sucesso um agente A2A com autenticação IAM, você pode:

1. **Adicionar mais ferramentas**: Estenda seu agente com ferramentas adicionais
2. **Sistemas multi-agente**: Crie agentes orquestradores que chamam este agente
3. **Políticas IAM personalizadas**: Crie políticas IAM granulares para acesso ao agente
4. **Integração**: Integre com outros serviços AWS

### Tutoriais relacionados

* [Hospedagem de servidor MCP com autenticação IAM](../02-hosting-MCP-server/hosting_mcp_server_iam_auth.ipynb)
* [Hospedagem A2A com autenticação JWT](../05-hosting-a2a/01-a2a-getting-started-agentcore-strands-pt.ipynb)
* [Sistemas multi-agente](../05-hosting-a2a/02-a2a-deploy-orchestrator-pt.ipynb)

## Limpeza (Opcional)

Se você quiser limpar os recursos criados durante este tutorial, execute as seguintes células:

In [ ]:
destroy_bedrock_agentcore(
    config_path=Path(".bedrock_agentcore.yaml")
)
print("✓ AgentCore Runtime resources deleted")

## Conclusão

Neste tutorial, você aprendeu como:

* Criar um agente A2A usando o framework Strands
* Implantar o agente no AgentCore Runtime
* Configurar autenticação IAM para requisições de entrada
* Testar o agente usando requisições assinadas com SigV4